# Preview SNI 300 g dari instance-crop v1

Notebook ini hanya membuat dan menampilkan data sintetis. **Tidak ada training.**

Empat preview dibuat: `source_empirical/A1`, `source_empirical/A2`, `defect_enriched/A1`, dan `defect_enriched/A2`. Normal tetap menjadi kelas terbanyak. `source_empirical` bukan klaim prevalensi dunia nyata.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, subprocess, sys

REPO_ROOT = Path('/content/coffee-bean-detection')
if not REPO_ROOT.is_dir():
    subprocess.run(['git', 'clone', '--branch', 'agent/add-vadcp-pipeline', 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
os.chdir(REPO_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
import coffee_detector
print('IMPORT BERHASIL:', coffee_detector.__file__)

In [ ]:
CROP_DATASET_ROOT = Path('/content/drive/MyDrive/coffee-sni-instance-crop-v1')
OUTPUT_ROOT = Path('/content/coffee-sni-300g-preview')
DRIVE_RESULT_ROOT = Path('/content/drive/MyDrive/coffee-bean-detection/sni-crop-300g-preview')

required = [CROP_DATASET_ROOT / 'manifest.csv', CROP_DATASET_ROOT / 'complete.json', CROP_DATASET_ROOT / 'shards']
for path in required:
    print(('ADA   ' if path.exists() else 'HILANG'), '-', path)
assert all(path.exists() for path in required), 'Dataset crop di Drive belum lengkap.'
print('OUTPUT LOKAL:', OUTPUT_ROOT)
print('TRAINING: TIDAK DIJALANKAN')

In [ ]:
# Progress tampil untuk setiap shard, setiap 100 mask, dan setiap scene.
cmd = [sys.executable, '-u', '-m', 'coffee_detector.run_sni_crop_preview',
       '--crop-dataset-root', str(CROP_DATASET_ROOT), '--output-root', str(OUTPUT_ROOT),
       '--images', '4', '--objects-min', '220', '--objects-max', '300',
       '--canvas-size', '1024', '--enriched-normal-fraction', '0.55',
       '--max-normal-assets', '300', '--max-defect-assets-per-class', '60',
       '--shard-cache-root', '/content/coffee-sni-shard-cache', '--seed', '42']
print('MULAI PREVIEW — BUKAN TRAINING', flush=True)
subprocess.run(cmd, check=True)
print('PREVIEW SELESAI', flush=True)

In [ ]:
import json
from IPython.display import Image as DisplayImage, display
summary = json.loads((OUTPUT_ROOT / 'preview_summary.json').read_text())
print('=== KOMPOSISI AKTUAL ===')
for arm, row in summary['compositions'].items():
    print(f"{arm:28s} normal={row['normal_fraction']:.2%} instances={row['labeled_instances']} repeated={row['repeated_assets']}")
print('\n=== CUTOUT EDGE AUDIT ===')
display(DisplayImage(filename=summary['cutout_contact_sheet']))
for arm, path in summary['raw_contact_sheets'].items():
    print('\n', arm)
    display(DisplayImage(filename=path))

In [ ]:
# Simpan ringkasan dan contact sheet ke Drive setelah review.
import shutil
DRIVE_RESULT_ROOT.mkdir(parents=True, exist_ok=True)
relatives = ['preview_summary.json', 'cutout_visual/contact_sheet.jpg',
             'source_empirical/A1_visual/contact_sheet_raw.jpg', 'source_empirical/A2_visual/contact_sheet_raw.jpg',
             'defect_enriched/A1_visual/contact_sheet_raw.jpg', 'defect_enriched/A2_visual/contact_sheet_raw.jpg']
for relative in relatives:
    source = OUTPUT_ROOT / relative
    target = DRIVE_RESULT_ROOT / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, target)
    print('SAVED:', target)
print('Training tetap belum dijalankan.')